In [17]:
# import packages
%pip install shap
%pip install shap matplotlib
import shap
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import shap
import matplotlib.pyplot as plt

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [6]:
seed = 2724

### Import data

In [3]:
DF_PATH = "mod04_data/sample.csv"
df = pd.read_csv(DF_PATH)

### Separate data by independent (X) and dependent (y) variables

In [4]:
X = df[["income", "education_years", "zipcode_score"]]
y = df["target"]

### Split the data into a _training_ set (to build a model) and _test_ set (to validate a model)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=seed
)

### Build a model on the training set

In [8]:
model = RandomForestRegressor(
    n_estimators=200,
    random_state=seed
)
model.fit(X_train, y_train)

RandomForestRegressor(n_estimators=200, random_state=2724)

### Use SHAP to explain the model on test data

In [15]:
explainer = shap.Explainer(model, X_train)
shap_values = explainer(X_test)

100%|===================| 1498/1500 [00:44<00:00]        

This will allow us to see which variables are most important to predicting the outcome.

In [10]:
shap.plots.bar(shap_values)

NameError: name 'shap' is not defined

### Import the `group` variable, which was **not** used in training this model.

In [ ]:
X_test_with_group = X_test.copy()
X_test_with_group["group"] = df.loc[X_test.index, "group"]

### Look at the difference in SHAP values between the two groups across the variables used in the model.

In [16]:
shap_df = pd.DataFrame(shap_values.values, columns=X_test.columns)
shap_df["group"] = X_test_with_group["group"].values

shap_df.groupby("group").mean()


NameError: name 'X_test_with_group' is not defined

### Let's put `group` and `zipcode_score` in the same plot

In [12]:
def plot_shap(var):
    # Extract SHAP values for the feature
    shap_var = shap_values[:, var].values

    # Plot the values of each group using different colors
    plt.figure()
    plt.scatter(
        X_test[var],
        shap_var,
        c=X_test_with_group["group"]
    )
    plt.xlabel(var)
    plt.ylabel(f"SHAP value for var")
    plt.title("Proxy feature impact by group")
    plt.show()

plot_shap("zipcode_score")

NameError: name 'shap_values' is not defined

# Discussion Questions

### What is a _SHAP_ (or Shapley) value? 

A SHAP value shows how much a feature changed the model's prediction, compared with the model's average (baseline) prediction. If the SHAP value is positive, the feature made the prediction go up. If it's negative, the feature made the prediction go down. And by looking at these values we can see which variables affect the model's predictions the most and whcih features didnot affect the prediction.

### Suppose you built this model and then it is peer reviewed by another entity. If the reviewer asks whether you used the variable `group` in your model, what would your answer be?

My answer would be No because the group variable was not used to train the model. The model only used income, education_years, and zipcode_score which are independent variables.

### If the reviewer asks whether the outcome of your model is correlated with `group`, what would your answer be?

Yes. Even though group was not used in the model, the predictions are still related to it. Group 0 had an average prediction of about 100.05, while Group 1 had an average of about 87.24. This tells that some of the other variables in the model may be indirectly representing the group variable.

### Construct a "proxy feature impact by group" plot for `income`. How is this plot different from the one for `zipcode_score`?

In [1]:
plot_shap("income")

NameError: name 'plot_shap' is not defined

The income plot shows that the two groups are more mixed together and follow a similar pattern. However, the zipcode score plot shows a clearer difference between the two groups. This means that income is not closely related to the group, while zipcode score seems to be more connected to the group and may be affecting the model's predictions.

### If, instead, you were the **reviewer**, what other questions might you ask the person who built this model? Give at least two.

As a reviewer, I would first ask if any of the variables in the model are closely related to group and could indirectly represent it. I would also ask if the model gives different predictions or has different error rates for the two groups. Finally, I would check if both groups are fairly represented in the training data and if removing a variable like zipcode_score changes the results.